# evaluation demo
Why a good-looking backtest isn't enough. We sweep 12 configurations **in-sample**, pick the best, and deflate its Sharpe for the search. Then we look at the hold-out once.

In [1]:
import itertools

from quantskills.data import synthetic_prices
from quantskills.eval import deflated_sharpe_ratio, tear_sheet
from quantskills.strategies import FactorLongShort, run_strategy, sweep_params
from quantskills.utils import holdout_date

prices = synthetic_prices(seed=0)
train_end = holdout_date(prices.index.get_level_values("date"), 0.3)
grid = [
    dict(quantile=q, rebalance_days=r, weighting=w)
    for q, r, w in itertools.product((0.1, 0.2, 0.3), (5, 21), ("ic", "equal"))
]
table = sweep_params(FactorLongShort, grid, prices, train_end)
table.sort_values("is_sharpe", ascending=False).round(3)

,quantile,rebalance_days,weighting,is_sharpe
2,0.1,21,ic,1.529
10,0.3,21,ic,1.480
6,0.2,21,ic,1.423
4,0.2,5,ic,1.266
8,0.3,5,ic,0.991
0,0.1,5,ic,0.984
7,0.2,21,equal,0.199
9,0.3,5,equal,-0.006
11,0.3,21,equal,-0.097
5,0.2,5,equal,-0.547


In [2]:
best = table.loc[table.is_sharpe.idxmax()].drop("is_sharpe").to_dict()
best["rebalance_days"] = int(best["rebalance_days"])
run = run_strategy(FactorLongShort(**best), prices, train_end)
d = deflated_sharpe_ratio(run.in_sample.trim_warmup().returns, trial_sharpes=table.is_sharpe)
d

DeflatedSharpe(sharpe=1.5287473339576922, sharpe_threshold=1.4284311711079467, psr=0.9920658882616546, dsr=0.5628782247165138, n_trials=12)

PSR says the in-sample Sharpe is very likely > 0. DSR asks a harder question: does it beat the best of 12 *useless* strategies? Now the hold-out, touched once:

In [3]:
s = tear_sheet(run.out_of_sample, trial_sharpes=table.is_sharpe)
{k: s[k] for k in ("sharpe", "ann_return", "max_drawdown", "dsr", "n_trials")}

{'sharpe': -1.006450747105671,
 'ann_return': -0.10249798419949296,
 'max_drawdown': -0.23211956562379976,
 'dsr': 0.0014075412024898526,
 'n_trials': 12}

On a pure-noise panel the same machinery should find nothing:

In [4]:
null = synthetic_prices(seed=0, drift_vol=0.0)
t = sweep_params(FactorLongShort, grid, null, train_end)
best_null = t.loc[t.is_sharpe.idxmax()].drop("is_sharpe").to_dict()
best_null["rebalance_days"] = int(best_null["rebalance_days"])
r = run_strategy(FactorLongShort(**best_null), null, train_end)
print(
    f"null best IS Sharpe {t.is_sharpe.max():.2f}, DSR {deflated_sharpe_ratio(r.in_sample.trim_warmup().returns, trial_sharpes=t.is_sharpe).dsr:.3f}"
)

null best IS Sharpe 1.42, DSR 0.652
